In [1]:
# ============================================================
# SECTION 1: INSTALL DEPENDENCIES
# ============================================================
#
# NOTE: Unlike a Colab VM, the Intel Unnati GPU Server has a persistent
# filesystem, so installed packages survive kernel restarts. We still use
# a flag file so repeated notebook runs don't re-invoke pip unnecessarily.

import os
import subprocess
import sys
from pathlib import Path

# Local (non-Colab) flag location — persists across kernel restarts on this server.
ENV_FLAG_PATH = Path.home() / ".calibrag_env_ready"


def run_pip_install(packages: list[str], quiet: bool = True) -> None:
    """
    Install a list of pip packages using the current Python interpreter.

    Args:
        packages: list of pip package specifiers (e.g. "torch==2.3.0")
        quiet: suppress verbose pip output if True

    Raises:
        RuntimeError: if pip installation fails for any package group
    """
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd.extend(packages)
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Failed to install packages: {packages}\nError: {e}")


if ENV_FLAG_PATH.exists():
    print("✓ Existing environment flag found — skipping dependency installation.")
else:
    print("→ No environment flag found. Installing dependencies...\n")

    print("[1/6] Core deep learning libraries (torch, transformers, accelerate)...")
    run_pip_install([
        "transformers>=4.42.0",
        "accelerate>=0.31.0",
        "bitsandbytes>=0.43.1",   # optional quantized loading, useful on smaller GPUs
        "sentencepiece",
    ])

    print("[2/6] Retrieval and embedding libraries (sentence-transformers, faiss)...")
    run_pip_install([
        "sentence-transformers>=3.0.1",
        "faiss-gpu-cu12",         # GPU FAISS build for CUDA — adjust CUDA suffix
                                   # to match the server's installed CUDA toolkit
                                   # (e.g. faiss-gpu-cu11) if this fails to import.
    ])

    print("[3/6] Dataset libraries (HuggingFace datasets)...")
    run_pip_install(["datasets>=2.20.0", "huggingface_hub>=0.23.0"])

    print("[4/6] Evaluation and calibration libraries (scikit-learn, scipy)...")
    run_pip_install(["scikit-learn>=1.5.0", "scipy>=1.13.0"])

    print("[5/6] Reporting and visualization libraries...")
    run_pip_install([
        "pandas>=2.2.0",
        "matplotlib>=3.9.0",
        "openpyxl>=3.1.0",
        "psutil>=5.9.0",   # for CPU/RAM diagnostics (Section 2)
    ])

    print("[6/6] Utilities (tqdm)...")
    run_pip_install(["tqdm>=4.66.0"])

    ENV_FLAG_PATH.write_text("environment_ready=true\n")
    print("\n✓ All dependencies installed successfully.")

# ------------------------------------------------------------
# Sanity check: verify critical imports work before proceeding
# ------------------------------------------------------------
print("\n→ Verifying critical package imports...")
try:
    import torch, transformers, sentence_transformers, faiss, datasets
    import sklearn, pandas, matplotlib, openpyxl, tqdm, psutil

    print("✓ torch:", torch.__version__)
    print("✓ transformers:", transformers.__version__)
    print("✓ sentence_transformers:", sentence_transformers.__version__)
    print("✓ faiss: OK")
    print("✓ datasets:", datasets.__version__)
    print("✓ All critical imports verified successfully.")
except ImportError as e:
    raise RuntimeError(
        f"Critical import failed after installation: {e}\n"
        f"Try deleting {ENV_FLAG_PATH} and re-running this cell."
    )

✓ Existing environment flag found — skipping dependency installation.

→ Verifying critical package imports...


/home/23adr021/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1785919045.257853 1619092 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785919045.322582 1619092 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785919054.220135 1619092 port.cc:153] oneDNN custom operations are on. You 

✓ torch: 2.7.0+cu126
✓ transformers: 4.57.6
✓ sentence_transformers: 5.6.1
✓ faiss: OK
✓ datasets: 5.0.1
✓ All critical imports verified successfully.


In [2]:
import transformers
import keras
import tensorflow as tf

print("Transformers:", transformers.__version__)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

Transformers: 4.57.6
TensorFlow: 2.21.0
Keras: 3.15.1


In [3]:
!pip install -U tf-keras

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# ============================================================
# SECTION 2: IMPORT LIBRARIES, SEEDING, HARDWARE DIAGNOSTICS
# ============================================================

import os
import json
import time
import random
import logging
import platform
import subprocess
import warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
import psutil

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from sentence_transformers import SentenceTransformer, CrossEncoder

import faiss
from datasets import load_dataset

from sklearn.metrics import f1_score, brier_score_loss
from scipy.stats import entropy as scipy_entropy

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("CalibRAG")


# ------------------------------------------------------------
# Global random seed control
# ------------------------------------------------------------
def set_global_seed(seed: int = 42) -> None:
    """
    Set random seeds across Python, NumPy, and PyTorch (CPU + CUDA) for
    reproducible results across notebook re-runs.

    Args:
        seed: integer seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info(f"Global random seed set to {seed}")


GLOBAL_SEED = 42
set_global_seed(GLOBAL_SEED)


# ------------------------------------------------------------
# Hardware / software diagnostics
# ------------------------------------------------------------
def _get_nvidia_smi_summary() -> Optional[str]:
    """Return a one-line nvidia-smi GPU summary, or None if unavailable."""
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"],
            text=True,
        ).strip()
        return output
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def detect_hardware() -> Dict[str, Any]:
    """
    Detect and report the full hardware/software environment: GPU, CUDA,
    CPU, RAM, and key library versions. Prints a formatted summary and
    returns the same information as a dict (used later to write
    configs/environment.json).

    Returns:
        Dict of environment fields.
    """
    gpu_available = torch.cuda.is_available()
    info: Dict[str, Any] = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": gpu_available,
        "cuda_version": torch.version.cuda if gpu_available else None,
        "transformers_version": __import__("transformers").__version__,
        "sentence_transformers_version": __import__("sentence_transformers").__version__,
        "datasets_version": __import__("datasets").__version__,
        "faiss_version": getattr(faiss, "__version__", "unknown"),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024 ** 3), 1),
        "gpu_name": None,
        "gpu_total_memory_gb": None,
        "nvidia_smi_summary": _get_nvidia_smi_summary(),
    }

    if gpu_available:
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_total_memory_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1
        )

    print("\n" + "=" * 60)
    print("SECTION 2 — HARDWARE / SOFTWARE ENVIRONMENT")
    print("=" * 60)
    print(f"Python               : {info['python_version']}")
    print(f"Platform             : {info['platform']}")
    print(f"Torch                : {info['torch_version']}")
    print(f"CUDA available       : {info['cuda_available']}")
    print(f"CUDA version         : {info['cuda_version']}")
    print(f"GPU                  : {info['gpu_name']} "
          f"({info['gpu_total_memory_gb']} GB VRAM)" if gpu_available else "GPU: none detected (CPU fallback)")
    print(f"CPU cores (logical)  : {info['cpu_count_logical']}")
    print(f"RAM total            : {info['ram_total_gb']} GB")
    print(f"Transformers         : {info['transformers_version']}")
    print(f"SentenceTransformers : {info['sentence_transformers_version']}")
    print(f"Datasets             : {info['datasets_version']}")
    print(f"FAISS                : {info['faiss_version']}")
    print("=" * 60 + "\n")

    if not gpu_available:
        logger.warning("No GPU detected — falling back to CPU. This will be slow "
                        "for the generator/NLI models used later in the pipeline.")

    return info


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HARDWARE_INFO = detect_hardware()

logger.info("✓ Section 2 (Imports, Seeding, Diagnostics) completed successfully.")

[2026-08-05 14:08:56] INFO: Global random seed set to 42
[2026-08-05 14:08:56] INFO: ✓ Section 2 (Imports, Seeding, Diagnostics) completed successfully.



SECTION 2 — HARDWARE / SOFTWARE ENVIRONMENT
Python               : 3.11.11
Platform             : Linux-6.8.0-124-generic-x86_64-with-glibc2.35
Torch                : 2.7.0+cu126
CUDA available       : True
CUDA version         : 12.6
GPU                  : NVIDIA L4 (22.0 GB VRAM)
CPU cores (logical)  : 64
RAM total            : 251.5 GB
Transformers         : 4.57.6
SentenceTransformers : 5.6.1
Datasets             : 5.0.1
FAISS                : 1.14.1



In [5]:
# ============================================================
# SECTION 3: CONFIGURATION, EXECUTION MODES, PROJECT STRUCTURE
# ============================================================

# ------------------------------------------------------------
# 3.1 — Execution mode (the ONLY place sample size is controlled)
# ------------------------------------------------------------
# Set RUN_MODE here. NUM_SAMPLES is derived from RUN_MODE by default,
# but you may override NUM_SAMPLES directly below for a custom subset
# size (e.g. 100, 250, 500) — no other cell in this notebook hardcodes
# a sample size.

RUN_MODE: str = "development"     # "development" or "research"
EXPERIMENT_NAME: str = "CalibRAG_v1"

_RUN_MODE_DEFAULTS = {
    "development": 50,
    "research": 1000,
}
if RUN_MODE not in _RUN_MODE_DEFAULTS:
    raise ValueError(f"RUN_MODE must be one of {list(_RUN_MODE_DEFAULTS)}, got {RUN_MODE!r}")

NUM_SAMPLES: Optional[int] = _RUN_MODE_DEFAULTS[RUN_MODE]
# ^ Override explicitly if needed, e.g.:
#   NUM_SAMPLES = 250
#   NUM_SAMPLES = None   # None == entire HotpotQA validation split


# ------------------------------------------------------------
# 3.2 — Project root and experiment-scoped directory structure
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd() / "CalibRAG_Project"
EXPERIMENT_ROOT = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME

SUBDIRS = [
    "checkpoints", "datasets", "embeddings", "faiss_index",
    "retrieval", "reranking", "baseline", "hypotheses", "claims",
    "nli_scores", "aggregation", "iterations", "answers",
    "evaluation", "metrics", "tables", "figures", "logs",
    "configs", "final_results",
]


def create_project_structure(root: Path, subdirs: List[str]) -> Dict[str, Path]:
    """
    Create the experiment directory tree if it does not already exist.
    Idempotent — safe to call on every notebook run.

    Args:
        root: root Path for this experiment (.../experiments/<name>)
        subdirs: subdirectory names to create under root

    Returns:
        Dict mapping subdirectory name -> full Path.
    """
    root.mkdir(parents=True, exist_ok=True)
    paths = {"root": root}
    for name in subdirs:
        p = root / name
        p.mkdir(parents=True, exist_ok=True)
        paths[name] = p
    return paths


PATHS = create_project_structure(EXPERIMENT_ROOT, SUBDIRS)

print(f"✓ Project root      : {PROJECT_ROOT}")
print(f"✓ Experiment root   : {EXPERIMENT_ROOT}")
print(f"✓ RUN_MODE          : {RUN_MODE}")
print(f"✓ NUM_SAMPLES       : {NUM_SAMPLES if NUM_SAMPLES is not None else 'FULL validation split'}")
for name in SUBDIRS:
    print(f"   ├── {name}/")


# ------------------------------------------------------------
# 3.3 — Central configuration dictionary
# ------------------------------------------------------------
# Every downstream section reads from CONFIG rather than hardcoding values.

CONFIG: Dict[str, Any] = {
    # ---- Experiment identity ----
    "experiment_name": EXPERIMENT_NAME,
    "run_mode": RUN_MODE,
    "random_seed": GLOBAL_SEED,
    "device": DEVICE,

    # ---- Dataset ----
    "dataset_name": "hotpotqa/hotpot_qa",
    "dataset_config": "distractor",
    "dataset_split": "validation",
    "num_samples": NUM_SAMPLES,          # None == full split; only place this is set

    # ---- Chunking ----
    "chunk_size_tokens": 150,
    "chunk_overlap_tokens": 20,

    # ---- Embedding / Retrieval ----
    "embedding_model": "sentence-transformers/all-mpnet-base-v2",
    "embedding_batch_size": 32,
    "top_k_retrieval": 8,
    "faiss_index_type": "IndexFlatIP",

    # ---- Cross-Encoder Reranking ----
    "cross_encoder_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "rerank_top_k": 4,

    # ---- Generator LLM (shared identically across ALL baselines) ----
    "generator_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "generator_max_new_tokens": 256,
    "generator_temperature": 0.3,
    "generator_load_in_4bit": True,

    # ---- NLI Verifier (CalibRAG core) ----
    "nli_model": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    "nli_batch_size": 16,

    # ---- Verification thresholds (CalibRAG) ----
    "entailment_confidence_threshold": 0.70,
    "claim_support_threshold": 0.60,
    "max_retrieval_rounds": 3,

    # ---- Shared retrieval-budget / stopping criteria (fair comparison across baselines) ----
    "retrieval_budget_max_calls": 3,     # max retrieval calls any baseline may issue per question
    "max_iterations": 3,                 # max reasoning/verification iterations any baseline may run

    # ---- Baseline-specific thresholds (kept explicit for auditability) ----
    "adakg_confidence_threshold": 0.70,          # same as CalibRAG's, for fairness
    "self_rag_relevance_threshold": 0.60,        # Self-RAG: min relevance to keep a passage
    "crag_correct_threshold": 0.60,              # CRAG: retrieval evaluator "Correct" cutoff
    "crag_incorrect_threshold": 0.30,            # CRAG: retrieval evaluator "Incorrect" cutoff
    "adaptive_rag_complexity_model": "generator", # Adaptive RAG: use the generator LLM itself
                                                   # as the query-complexity classifier (no
                                                   # separately trained classifier available)

    # ---- Checkpointing ----
    "checkpoint_every_n_questions": 25,

    # ---- Statistical testing ----
    "significance_test": "paired_bootstrap",
    "significance_n_bootstrap": 10000,
    "significance_alpha": 0.05,

    # ---- Output directories ----
    "paths": {k: str(v) for k, v in PATHS.items()},

    # ---- Config metadata ----
    "config_version": "2.0",
    "last_updated": datetime.now().isoformat(),
}

print(f"\n✓ CONFIG initialized with {len(CONFIG)} top-level keys.")


# ------------------------------------------------------------
# 3.4 — experiment_state.json: checkpoint tracking + config validation
# ------------------------------------------------------------
STATE_FILE_PATH = EXPERIMENT_ROOT / "experiment_state.json"

SECTION_NAMES = [
    "install_dependencies", "import_libraries", "configuration",
    "load_dataset", "context_processing", "embedding_generation",
    "dense_retrieval", "cross_encoder_reranking",
    "baseline_vanilla_rag", "baseline_self_rag", "baseline_crag",
    "baseline_adaptive_rag", "baseline_adakg_rag",
    "hypothesis_generation", "claim_decomposition", "claim_level_nli_verification",
    "sufficiency_aggregation", "verification_guided_retrieval_loop",
    "final_answer_generation", "supporting_fact_ranking", "evaluation_pipeline",
    "accuracy_metrics", "calibration_metrics", "efficiency_metrics",
    "robustness_metrics", "statistical_significance", "budget_matched_ablation",
    "visualization", "final_summary",
]

# Config fields that, if changed, invalidate existing checkpoints for THIS
# experiment name (mixing incompatible experiments is what we're guarding against).
CHECKPOINT_AFFECTING_KEYS = [
    "dataset_name", "dataset_config", "dataset_split", "num_samples",
    "chunk_size_tokens", "chunk_overlap_tokens",
    "embedding_model", "top_k_retrieval",
    "cross_encoder_model", "rerank_top_k",
    "generator_model", "nli_model",
    "entailment_confidence_threshold", "claim_support_threshold",
    "max_retrieval_rounds", "random_seed",
]


def build_fresh_state(config: Dict[str, Any]) -> Dict[str, Any]:
    """Construct a brand-new experiment_state.json structure."""
    return {
        "experiment_name": config["experiment_name"],
        "created_at": datetime.now().isoformat(),
        "last_updated": datetime.now().isoformat(),
        "config_snapshot": config,
        "sections": {name: {"completed": False, "timestamp": None} for name in SECTION_NAMES},
        "resume_pointers": {},   # section_key -> last fully-checkpointed question index
    }


def config_keys_that_affect_checkpoints(config: Dict[str, Any]) -> Dict[str, Any]:
    """Extract the checkpoint-relevant subset of CONFIG for equality comparison."""
    return {k: config.get(k) for k in CHECKPOINT_AFFECTING_KEYS}


def save_experiment_state() -> None:
    """Persist EXPERIMENT_STATE to disk. Called after every section completes."""
    EXPERIMENT_STATE["last_updated"] = datetime.now().isoformat()
    with open(STATE_FILE_PATH, "w") as f:
        json.dump(EXPERIMENT_STATE, f, indent=2, default=str)


def mark_section_complete(section_key: str) -> None:
    """Mark a section complete in EXPERIMENT_STATE and persist immediately."""
    EXPERIMENT_STATE["sections"].setdefault(section_key, {})
    EXPERIMENT_STATE["sections"][section_key]["completed"] = True
    EXPERIMENT_STATE["sections"][section_key]["timestamp"] = datetime.now().isoformat()
    save_experiment_state()


def is_section_complete(section_key: str) -> bool:
    """Check whether a given section has already been completed."""
    return EXPERIMENT_STATE["sections"].get(section_key, {}).get("completed", False)


def set_resume_pointer(section_key: str, last_index: int) -> None:
    """Record the last fully-checkpointed question index for a section (Section-13-style resume)."""
    EXPERIMENT_STATE.setdefault("resume_pointers", {})[section_key] = last_index
    save_experiment_state()


def get_resume_pointer(section_key: str) -> int:
    """Return the last fully-checkpointed question index for a section, or -1 if none."""
    return EXPERIMENT_STATE.get("resume_pointers", {}).get(section_key, -1)


if STATE_FILE_PATH.exists():
    logger.info("Existing experiment_state.json found for this experiment. Loading...")
    with open(STATE_FILE_PATH, "r") as f:
        EXPERIMENT_STATE = json.load(f)

    old_relevant = config_keys_that_affect_checkpoints(EXPERIMENT_STATE.get("config_snapshot", {}))
    new_relevant = config_keys_that_affect_checkpoints(CONFIG)

    if old_relevant != new_relevant:
        print("\n" + "!" * 60)
        print("⚠ WARNING: Configuration has changed since the last run of "
              f"'{EXPERIMENT_NAME}'!")
        print("!" * 60)
        for key in new_relevant:
            if old_relevant.get(key) != new_relevant.get(key):
                print(f"  • {key}: OLD={old_relevant.get(key)!r}  →  NEW={new_relevant.get(key)!r}")
        print(
            "\nReusing old checkpoints under this changed config may mix incompatible "
            "artifacts (e.g. embeddings from one model with an index built by another).\n"
            "Recommended: change EXPERIMENT_NAME in Section 3.1 to start a fresh "
            "experiment, or revert CONFIG to match the previous run.\n"
        )
        print("!" * 60 + "\n")
    else:
        logger.info("✓ Configuration matches the previous run of this experiment. "
                     "Checkpoints are valid for reuse.")

    EXPERIMENT_STATE["config_snapshot"] = CONFIG
else:
    logger.info(f"No experiment_state.json found for '{EXPERIMENT_NAME}'. Creating a fresh one.")
    EXPERIMENT_STATE = build_fresh_state(CONFIG)

save_experiment_state()
mark_section_complete("configuration")


# ------------------------------------------------------------
# 3.5 — Save config snapshot + environment.json for reproducibility
# ------------------------------------------------------------
config_snapshot_path = PATHS["configs"] / f"{CONFIG['experiment_name']}_config.json"
with open(config_snapshot_path, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

environment_log = {
    **HARDWARE_INFO,
    "random_seed": GLOBAL_SEED,
    "generator_model": CONFIG["generator_model"],
    "embedding_model": CONFIG["embedding_model"],
    "cross_encoder_model": CONFIG["cross_encoder_model"],
    "nli_model": CONFIG["nli_model"],
    "execution_date": datetime.now().isoformat(),
    "experiment_name": CONFIG["experiment_name"],
    "run_mode": CONFIG["run_mode"],
}
with open(PATHS["configs"] / "environment.json", "w") as f:
    json.dump(environment_log, f, indent=2, default=str)

logger.info(f"✓ Config snapshot saved to {config_snapshot_path}")
logger.info(f"✓ Environment log saved to {PATHS['configs'] / 'environment.json'}")

# ------------------------------------------------------------
# Final printout
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("SECTION 3 COMPLETE — CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Experiment name       : {CONFIG['experiment_name']}")
print(f"Run mode / N samples  : {CONFIG['run_mode']} / {CONFIG['num_samples']}")
print(f"Dataset               : {CONFIG['dataset_name']} ({CONFIG['dataset_config']})")
print(f"Embedding model       : {CONFIG['embedding_model']}")
print(f"Cross-encoder model   : {CONFIG['cross_encoder_model']}")
print(f"Generator model       : {CONFIG['generator_model']}")
print(f"NLI model             : {CONFIG['nli_model']}")
print(f"Device                : {CONFIG['device']}")
print(f"Random seed           : {CONFIG['random_seed']}")
print("=" * 60 + "\n")

[2026-08-05 14:12:47] INFO: No experiment_state.json found for 'CalibRAG_v1'. Creating a fresh one.
[2026-08-05 14:12:47] INFO: ✓ Config snapshot saved to /home/23adr021/CalibRAG_Project/experiments/CalibRAG_v1/configs/CalibRAG_v1_config.json
[2026-08-05 14:12:47] INFO: ✓ Environment log saved to /home/23adr021/CalibRAG_Project/experiments/CalibRAG_v1/configs/environment.json


✓ Project root      : /home/23adr021/CalibRAG_Project
✓ Experiment root   : /home/23adr021/CalibRAG_Project/experiments/CalibRAG_v1
✓ RUN_MODE          : development
✓ NUM_SAMPLES       : 50
   ├── checkpoints/
   ├── datasets/
   ├── embeddings/
   ├── faiss_index/
   ├── retrieval/
   ├── reranking/
   ├── baseline/
   ├── hypotheses/
   ├── claims/
   ├── nli_scores/
   ├── aggregation/
   ├── iterations/
   ├── answers/
   ├── evaluation/
   ├── metrics/
   ├── tables/
   ├── figures/
   ├── logs/
   ├── configs/
   ├── final_results/

✓ CONFIG initialized with 39 top-level keys.

SECTION 3 COMPLETE — CONFIGURATION SUMMARY
Experiment name       : CalibRAG_v1
Run mode / N samples  : development / 50
Dataset               : hotpotqa/hotpot_qa (distractor)
Embedding model       : sentence-transformers/all-mpnet-base-v2
Cross-encoder model   : cross-encoder/ms-marco-MiniLM-L-6-v2
Generator model       : Qwen/Qwen2.5-1.5B-Instruct
NLI model             : MoritzLaurer/DeBERTa-v3-base-mnl

In [6]:
# ============================================================
# SECTION 4: LOAD DATASET
# ============================================================

import pickle

DATASET_CHECKPOINT_PATH = PATHS["datasets"] / "sample_dataset.pkl"


def load_and_sample_hotpotqa(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Load HotpotQA (distractor setting) from HuggingFace `datasets` and
    sample a fixed-size, seeded subset of questions.

    Args:
        config: global CONFIG dictionary (uses dataset_name, dataset_config,
                 dataset_split, num_samples, random_seed). num_samples=None
                 means the entire split is used.

    Returns:
        A list of dicts, one per sampled question, each containing:
            question_id, question, answer, context (list of [title, sentences]),
            supporting_facts (list of [title, sent_id]), type, level.
    """
    logger.info(
        f"Downloading dataset: {config['dataset_name']} "
        f"({config['dataset_config']}) split={config['dataset_split']} ..."
    )

    raw_dataset = load_dataset(
        config["dataset_name"],
        config["dataset_config"],
        split=config["dataset_split"],
    )

    total_available = len(raw_dataset)
    requested = config["num_samples"]
    subset_size = total_available if requested is None else min(requested, total_available)

    logger.info(
        f"Dataset loaded: {total_available} total examples available. "
        f"Sampling {subset_size} examples with seed={config['random_seed']}."
    )

    # Seeded shuffle for reproducible sampling across notebook re-runs and
    # across different num_samples values (smaller runs are a strict prefix
    # of larger runs under the same seed).
    shuffled_indices = list(range(total_available))
    rng = random.Random(config["random_seed"])
    rng.shuffle(shuffled_indices)
    selected_indices = shuffled_indices[:subset_size]

    sampled_data = []
    for idx in tqdm(selected_indices, desc="Processing sampled questions"):
        example = raw_dataset[idx]
        record = {
            "question_id": example.get("id", f"q_{idx:05d}"),
            "question": example["question"],
            "answer": example["answer"],
            "context": list(zip(example["context"]["title"], example["context"]["sentences"])),
            "supporting_facts": list(
                zip(example["supporting_facts"]["title"], example["supporting_facts"]["sent_id"])
            ),
            "type": example.get("type", "unknown"),
            "level": example.get("level", "unknown"),
        }
        sampled_data.append(record)

    return sampled_data


def save_dataset_checkpoint(data: List[Dict[str, Any]], path: Path) -> None:
    """Persist the sampled dataset to disk as a pickle file."""
    with open(path, "wb") as f:
        pickle.dump(data, f)
    logger.info(f"✓ Dataset checkpoint saved to {path}")


def load_dataset_checkpoint(path: Path) -> List[Dict[str, Any]]:
    """Load a previously saved sampled dataset pickle file."""
    with open(path, "rb") as f:
        return pickle.load(f)


# ------------------------------------------------------------
# Checkpoint detection and execution
# ------------------------------------------------------------
if DATASET_CHECKPOINT_PATH.exists() and is_section_complete("load_dataset"):
    print("✓ Existing checkpoint found. Loading previous results...")
    SAMPLED_DATASET = load_dataset_checkpoint(DATASET_CHECKPOINT_PATH)
    logger.info(f"Loaded {len(SAMPLED_DATASET)} cached questions from checkpoint.")
else:
    logger.info("No valid checkpoint found. Running dataset loading + sampling...")
    SAMPLED_DATASET = load_and_sample_hotpotqa(CONFIG)
    save_dataset_checkpoint(SAMPLED_DATASET, DATASET_CHECKPOINT_PATH)
    mark_section_complete("load_dataset")
    print("✓ Checkpoint saved successfully.")

# ------------------------------------------------------------
# Sanity checks + intermediate output
# ------------------------------------------------------------
assert len(SAMPLED_DATASET) > 0, "Sampled dataset is empty — check dataset loading logic."
assert all("question" in r and "answer" in r for r in SAMPLED_DATASET), \
    "Malformed dataset records detected."

type_counts = pd.Series([r["type"] for r in SAMPLED_DATASET]).value_counts()
level_counts = pd.Series([r["level"] for r in SAMPLED_DATASET]).value_counts()

print("\n" + "=" * 60)
print("SECTION 4 COMPLETE — DATASET SUMMARY")
print("=" * 60)
print(f"Total sampled questions : {len(SAMPLED_DATASET)}")
print("\nQuestion type distribution:")
print(type_counts.to_string())
print("\nQuestion difficulty distribution:")
print(level_counts.to_string())

example = SAMPLED_DATASET[0]
print("\n--- Example record (index 0) ---")
print(f"Question ID       : {example['question_id']}")
print(f"Question          : {example['question']}")
print(f"Gold Answer       : {example['answer']}")
print(f"# Context docs    : {len(example['context'])}")
print(f"# Supporting facts: {len(example['supporting_facts'])}")
print("=" * 60 + "\n")

[2026-08-05 14:14:16] INFO: No valid checkpoint found. Running dataset loading + sampling...
[2026-08-05 14:14:16] INFO: Downloading dataset: hotpotqa/hotpot_qa (distractor) split=validation ...
Generating validation split: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 7405/7405 [00:00<00:00, 12283.97 examples/s]
[2026-08-05 14:14:42] INFO: Dataset loaded: 7405 total examples available. Sampling 50 examples with seed=42.
Processing sampled questions: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 499.25it/s]
[2026-08-05 14:14:42] INFO: ✓ Dataset checkpoint saved to /home/23adr021/CalibRAG_Project/experiments/CalibRAG_v1/datasets/sample_dataset.pkl


✓ Checkpoint saved successfully.

SECTION 4 COMPLETE — DATASET SUMMARY
Total sampled questions : 50

Question type distribution:
bridge        41
comparison     9

Question difficulty distribution:
hard    50

--- Example record (index 0) ---
Question ID       : 5a7a567255429941d65f25bd
Question          : What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold Answer       : flotilla
# Context docs    : 10
# Supporting facts: 2



In [7]:
# ============================================================
# SECTION 5: CONTEXT PROCESSING (CHUNKING)
# ============================================================

CHUNKS_CHECKPOINT_PATH = PATHS["checkpoints"] / "chunks.pkl"


def whitespace_tokenize(text: str) -> List[str]:
    """Approximate tokenization by whitespace splitting (fast, model-agnostic)."""
    return text.split()


def chunk_document(
    title: str,
    sentences: List[str],
    chunk_size_tokens: int,
    overlap_tokens: int,
) -> List[Dict[str, Any]]:
    """
    Split a single document's sentences into overlapping token-count-based
    chunks, while preserving which original sentence indices each chunk spans.

    Args:
        title: document title (used as a stable identifier)
        sentences: list of sentence strings belonging to this document
        chunk_size_tokens: approximate target chunk size in whitespace tokens
        overlap_tokens: approximate token overlap between consecutive chunks

    Returns:
        List of dicts, each with "text" (chunk text) and "sentence_ids"
        (original sentence indices included in the chunk).
    """
    token_sentence_pairs: List[Tuple[str, int]] = []
    for sent_id, sentence in enumerate(sentences):
        for token in whitespace_tokenize(sentence):
            token_sentence_pairs.append((token, sent_id))

    if not token_sentence_pairs:
        return []

    chunks = []
    start = 0
    n_tokens = len(token_sentence_pairs)

    while start < n_tokens:
        end = min(start + chunk_size_tokens, n_tokens)
        window = token_sentence_pairs[start:end]
        chunk_tokens = [tok for tok, _ in window]
        chunk_sentence_ids = sorted(set(sid for _, sid in window))

        chunks.append({
            "text": " ".join(chunk_tokens),
            "sentence_ids": chunk_sentence_ids,
        })

        if end == n_tokens:
            break
        start = end - overlap_tokens

    return chunks


def process_all_contexts(
    dataset: List[Dict[str, Any]],
    config: Dict[str, Any],
) -> Tuple[List[str], List[Dict[str, Any]]]:
    """
    Chunk every context document for every question in the sampled dataset.

    Args:
        dataset: SAMPLED_DATASET (list of question records)
        config: global CONFIG dictionary

    Returns:
        chunk_texts: flat list of chunk text strings (parallel to chunk_metadata)
        chunk_metadata: flat list of metadata dicts (chunk_id, chunk_global_index,
            question_id, doc_title, sentence_ids)
    """
    chunk_texts: List[str] = []
    chunk_metadata: List[Dict[str, Any]] = []
    global_chunk_counter = 0

    for record in tqdm(dataset, desc="Chunking context documents"):
        question_id = record["question_id"]
        for doc_title, sentences in record["context"]:
            doc_chunks = chunk_document(
                title=doc_title,
                sentences=sentences,
                chunk_size_tokens=config["chunk_size_tokens"],
                overlap_tokens=config["chunk_overlap_tokens"],
            )
            for local_idx, chunk in enumerate(doc_chunks):
                chunk_id = f"{question_id}__{doc_title}__{local_idx}"
                chunk_texts.append(chunk["text"])
                chunk_metadata.append({
                    "chunk_id": chunk_id,
                    "chunk_global_index": global_chunk_counter,
                    "question_id": question_id,
                    "doc_title": doc_title,
                    "sentence_ids": chunk["sentence_ids"],
                })
                global_chunk_counter += 1

    return chunk_texts, chunk_metadata


def save_chunks_checkpoint(chunk_texts: List[str], chunk_metadata: List[Dict[str, Any]], path: Path) -> None:
    """Persist chunk texts + metadata to a single pickle file."""
    with open(path, "wb") as f:
        pickle.dump({"chunk_texts": chunk_texts, "chunk_metadata": chunk_metadata}, f)
    logger.info(f"✓ Chunks checkpoint saved to {path}")


def load_chunks_checkpoint(path: Path) -> Tuple[List[str], List[Dict[str, Any]]]:
    """Load previously saved chunk texts + metadata."""
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["chunk_texts"], data["chunk_metadata"]


# ------------------------------------------------------------
# Checkpoint detection and execution
# ------------------------------------------------------------
if CHUNKS_CHECKPOINT_PATH.exists() and is_section_complete("context_processing"):
    print("✓ Existing checkpoint found. Loading previous results...")
    CHUNK_TEXTS, CHUNK_METADATA = load_chunks_checkpoint(CHUNKS_CHECKPOINT_PATH)
    logger.info(f"Loaded {len(CHUNK_TEXTS)} cached chunks from checkpoint.")
else:
    logger.info("No valid checkpoint found. Running context chunking...")
    CHUNK_TEXTS, CHUNK_METADATA = process_all_contexts(SAMPLED_DATASET, CONFIG)
    save_chunks_checkpoint(CHUNK_TEXTS, CHUNK_METADATA, CHUNKS_CHECKPOINT_PATH)
    mark_section_complete("context_processing")
    print("✓ Checkpoint saved successfully.")

# ------------------------------------------------------------
# Sanity checks + intermediate output
# ------------------------------------------------------------
assert len(CHUNK_TEXTS) == len(CHUNK_METADATA), "Mismatch between chunk texts and metadata."
assert len(CHUNK_TEXTS) > 0, "No chunks were generated — check chunking logic."

chunks_per_question = pd.Series([m["question_id"] for m in CHUNK_METADATA]).value_counts()
chunk_lengths = [len(whitespace_tokenize(t)) for t in CHUNK_TEXTS]

print("\n" + "=" * 60)
print("SECTION 5 COMPLETE — CHUNKING SUMMARY")
print("=" * 60)
print(f"Total chunks generated       : {len(CHUNK_TEXTS)}")
print(f"Avg chunks per question      : {chunks_per_question.mean():.2f}")
print(f"Min / Max chunks per question: {chunks_per_question.min()} / {chunks_per_question.max()}")
print(f"Avg chunk length (tokens)    : {np.mean(chunk_lengths):.1f}")
print(f"Max chunk length (tokens)    : {np.max(chunk_lengths)}")
print("=" * 60 + "\n")

[2026-08-05 14:15:46] INFO: No valid checkpoint found. Running context chunking...
Chunking context documents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 3164.18it/s]
[2026-08-05 14:15:46] INFO: ✓ Chunks checkpoint saved to /home/23adr021/CalibRAG_Project/experiments/CalibRAG_v1/checkpoints/chunks.pkl


✓ Checkpoint saved successfully.

SECTION 5 COMPLETE — CHUNKING SUMMARY
Total chunks generated       : 547
Avg chunks per question      : 10.94
Min / Max chunks per question: 10 / 14
Avg chunk length (tokens)    : 81.2
Max chunk length (tokens)    : 150



In [8]:
# ============================================================
# SECTION 6: EMBEDDING GENERATION
# ============================================================

EMBEDDINGS_PATH = PATHS["embeddings"] / "chunk_embeddings.npy"
FAISS_INDEX_PATH = PATHS["faiss_index"] / "faiss.index"
CHUNK_MAPPING_PATH = PATHS["embeddings"] / "chunk_mapping.pkl"


def load_embedding_model(config: Dict[str, Any]) -> SentenceTransformer:
    """
    Load the configured sentence-transformers bi-encoder model onto the
    active device.

    Args:
        config: global CONFIG dictionary (uses embedding_model, device)

    Returns:
        A loaded SentenceTransformer model instance.
    """
    logger.info(f"Loading embedding model: {config['embedding_model']} ...")
    return SentenceTransformer(config["embedding_model"], device=config["device"])


def generate_chunk_embeddings(chunk_texts: List[str], model: SentenceTransformer, batch_size: int) -> np.ndarray:
    """
    Encode all chunk texts into L2-normalized embedding vectors.

    Args:
        chunk_texts: list of chunk text strings
        model: loaded SentenceTransformer model
        batch_size: encoding batch size

    Returns:
        np.ndarray of shape (n_chunks, embedding_dim), L2-normalized (float32).
    """
    embeddings = model.encode(
        chunk_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # required so inner product == cosine similarity
    )
    return embeddings.astype("float32")


def build_faiss_index(embeddings: np.ndarray, index_type: str) -> faiss.Index:
    """
    Build a FAISS index over the given embedding matrix.

    Args:
        embeddings: (n_chunks, dim) float32 embedding matrix
        index_type: currently supports "IndexFlatIP" (exact inner-product search)

    Returns:
        A populated FAISS index.
    """
    dim = embeddings.shape[1]
    if index_type != "IndexFlatIP":
        raise ValueError(f"Unsupported faiss_index_type: {index_type}")
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


def build_question_row_mapping(chunk_metadata: List[Dict[str, Any]]) -> Dict[str, List[int]]:
    """
    Build a mapping from question_id -> list of FAISS row indices belonging
    to that question's chunks, enabling per-question scoped retrieval.

    Args:
        chunk_metadata: CHUNK_METADATA list (parallel to embedding rows)

    Returns:
        Dict mapping question_id -> list of row indices.
    """
    mapping: Dict[str, List[int]] = {}
    for meta in chunk_metadata:
        mapping.setdefault(meta["question_id"], []).append(meta["chunk_global_index"])
    return mapping


def save_embedding_artifacts(
    embeddings: np.ndarray,
    index: faiss.Index,
    chunk_metadata: List[Dict[str, Any]],
    question_row_map: Dict[str, List[int]],
) -> None:
    """Persist embeddings, FAISS index, and chunk/question mapping to disk."""
    np.save(EMBEDDINGS_PATH, embeddings)
    faiss.write_index(index, str(FAISS_INDEX_PATH))
    with open(CHUNK_MAPPING_PATH, "wb") as f:
        pickle.dump({"chunk_metadata": chunk_metadata, "question_row_map": question_row_map}, f)
    logger.info("✓ Embedding artifacts saved: chunk_embeddings.npy, faiss.index, chunk_mapping.pkl")


def load_embedding_artifacts() -> Tuple[np.ndarray, faiss.Index, List[Dict[str, Any]], Dict[str, List[int]]]:
    """Load previously saved embeddings, FAISS index, and chunk/question mapping."""
    embeddings = np.load(EMBEDDINGS_PATH)
    index = faiss.read_index(str(FAISS_INDEX_PATH))
    with open(CHUNK_MAPPING_PATH, "rb") as f:
        mapping_data = pickle.load(f)
    return embeddings, index, mapping_data["chunk_metadata"], mapping_data["question_row_map"]


# ------------------------------------------------------------
# Checkpoint detection and execution
# ------------------------------------------------------------
all_embedding_files_exist = (
    EMBEDDINGS_PATH.exists() and FAISS_INDEX_PATH.exists() and CHUNK_MAPPING_PATH.exists()
)

if all_embedding_files_exist and is_section_complete("embedding_generation"):
    print("✓ Existing checkpoint found. Loading previous results...")
    CHUNK_EMBEDDINGS, FAISS_INDEX, CHUNK_METADATA_FROM_EMB, QUESTION_ROW_MAP = load_embedding_artifacts()
    logger.info(f"Loaded {CHUNK_EMBEDDINGS.shape[0]} cached embeddings from checkpoint.")
else:
    logger.info("No valid checkpoint found. Generating embeddings + building FAISS index...")
    embedding_model = load_embedding_model(CONFIG)
    CHUNK_EMBEDDINGS = generate_chunk_embeddings(CHUNK_TEXTS, embedding_model, CONFIG["embedding_batch_size"])
    FAISS_INDEX = build_faiss_index(CHUNK_EMBEDDINGS, CONFIG["faiss_index_type"])
    QUESTION_ROW_MAP = build_question_row_mapping(CHUNK_METADATA)
    CHUNK_METADATA_FROM_EMB = CHUNK_METADATA

    save_embedding_artifacts(CHUNK_EMBEDDINGS, FAISS_INDEX, CHUNK_METADATA, QUESTION_ROW_MAP)
    mark_section_complete("embedding_generation")
    print("✓ Checkpoint saved successfully.")

    del embedding_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# Sanity checks + intermediate output
# ------------------------------------------------------------
assert CHUNK_EMBEDDINGS.shape[0] == len(CHUNK_TEXTS), "Embedding count mismatch with chunk count."
assert FAISS_INDEX.ntotal == CHUNK_EMBEDDINGS.shape[0], "FAISS index size mismatch."
assert len(QUESTION_ROW_MAP) == len(SAMPLED_DATASET), "Question row map missing some questions."

print("\n" + "=" * 60)
print("SECTION 6 COMPLETE — EMBEDDING SUMMARY")
print("=" * 60)
print(f"Embedding model         : {CONFIG['embedding_model']}")
print(f"Total embeddings        : {CHUNK_EMBEDDINGS.shape[0]}")
print(f"Embedding dimension     : {CHUNK_EMBEDDINGS.shape[1]}")
print(f"FAISS index size        : {FAISS_INDEX.ntotal}")
print(f"Questions in row map    : {len(QUESTION_ROW_MAP)}")
print("=" * 60 + "\n")

[2026-08-05 14:16:06] INFO: No valid checkpoint found. Generating embeddings + building FAISS index...
[2026-08-05 14:16:06] INFO: Loading embedding model: sentence-transformers/all-mpnet-base-v2 ...
[2026-08-05 14:16:07] INFO: Loading SentenceTransformer model from sentence-transformers/all-mpnet-base-v2.
Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:06<00:00,  2.59it/s]
[2026-08-05 14:16:32] INFO: ✓ Embedding artifacts saved: chunk_embeddings.npy, faiss.index, chunk_mapping.pkl


✓ Checkpoint saved successfully.

SECTION 6 COMPLETE — EMBEDDING SUMMARY
Embedding model         : sentence-transformers/all-mpnet-base-v2
Total embeddings        : 547
Embedding dimension     : 768
FAISS index size        : 547
Questions in row map    : 50



In [9]:
# ============================================================
# SECTION 7: DENSE RETRIEVAL
# ============================================================

def retrieve_top_k_for_question(
    question: str,
    question_id: str,
    faiss_index: faiss.Index,
    embedding_model: SentenceTransformer,
    question_row_map: Dict[str, List[int]],
    chunk_metadata: List[Dict[str, Any]],
    chunk_texts: List[str],
    top_k: int,
) -> Dict[str, Any]:
    """
    Retrieve the top-k most similar chunks for a single question, scoped
    only to that question's own context chunks.

    Args:
        question: the question text to embed and search with
        question_id: HotpotQA question identifier
        faiss_index: the global FAISS IndexFlatIP built in Section 6
        embedding_model: loaded SentenceTransformer bi-encoder
        question_row_map: question_id -> list of valid FAISS row indices
        chunk_metadata: row_index -> metadata (parallel list to embeddings)
        chunk_texts: row_index -> chunk text (parallel list to embeddings)
        top_k: number of chunks to retrieve

    Returns:
        Dict with question_id, question, and a list of retrieved chunk
        records (chunk_id, text, similarity_score, metadata), sorted
        descending by similarity.

    Note:
        This function is the SINGLE retrieval implementation shared by
        every baseline (Vanilla RAG, Self-RAG, CRAG, Adaptive RAG,
        AdaKG-RAG, CalibRAG) — required for fair comparison.
    """
    query_embedding = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True,
    ).astype("float32")

    valid_rows = question_row_map[question_id]
    k_effective = min(top_k, len(valid_rows))

    id_selector = faiss.IDSelectorArray(np.array(valid_rows, dtype="int64"))
    search_params = faiss.SearchParameters(sel=id_selector)

    scores, indices = faiss_index.search(query_embedding, k_effective, params=search_params)
    scores, indices = scores[0], indices[0]

    retrieved_chunks = []
    for score, row_idx in zip(scores, indices):
        if row_idx == -1:
            continue  # FAISS pads with -1 if fewer than k results found
        meta = chunk_metadata[row_idx]
        retrieved_chunks.append({
            "chunk_id": meta["chunk_id"],
            "doc_title": meta["doc_title"],
            "sentence_ids": meta["sentence_ids"],
            "text": chunk_texts[row_idx],
            "similarity_score": float(score),
            "row_index": int(row_idx),
        })

    retrieved_chunks.sort(key=lambda x: x["similarity_score"], reverse=True)
    return {"question_id": question_id, "question": question, "retrieved_chunks": retrieved_chunks}


def save_retrieval_result(result: Dict[str, Any], path: Path) -> None:
    """Persist a single question's retrieval result to a pickle file."""
    with open(path, "wb") as f:
        pickle.dump(result, f)


def load_retrieval_result(path: Path) -> Dict[str, Any]:
    """Load a previously saved single-question retrieval result."""
    with open(path, "rb") as f:
        return pickle.load(f)


# ------------------------------------------------------------
# Load embedding model only if work remains
# ------------------------------------------------------------
retrieval_dir = PATHS["retrieval"]
questions_needing_retrieval = [
    r for r in SAMPLED_DATASET if not (retrieval_dir / f"{r['question_id']}.pkl").exists()
]

if not questions_needing_retrieval and is_section_complete("dense_retrieval"):
    print("✓ Existing checkpoint found for all questions.")
    embedding_model_for_retrieval = None
else:
    logger.info(f"{len(questions_needing_retrieval)} / {len(SAMPLED_DATASET)} questions need retrieval.")
    embedding_model_for_retrieval = SentenceTransformer(CONFIG["embedding_model"], device=CONFIG["device"])

# ------------------------------------------------------------
# Run retrieval per-question, with per-question checkpointing
# and periodic resume-pointer updates (every checkpoint_every_n_questions)
# ------------------------------------------------------------
RETRIEVAL_RESULTS: Dict[str, Dict[str, Any]] = {}
checkpoint_every = CONFIG["checkpoint_every_n_questions"]

for i, record in enumerate(tqdm(SAMPLED_DATASET, desc="Dense retrieval per question")):
    qid = record["question_id"]
    result_path = retrieval_dir / f"{qid}.pkl"

    if result_path.exists():
        result = load_retrieval_result(result_path)
    else:
        result = retrieve_top_k_for_question(
            question=record["question"], question_id=qid,
            faiss_index=FAISS_INDEX, embedding_model=embedding_model_for_retrieval,
            question_row_map=QUESTION_ROW_MAP, chunk_metadata=CHUNK_METADATA_FROM_EMB,
            chunk_texts=CHUNK_TEXTS, top_k=CONFIG["top_k_retrieval"],
        )
        save_retrieval_result(result, result_path)

    RETRIEVAL_RESULTS[qid] = result

    if (i + 1) % checkpoint_every == 0:
        set_resume_pointer("dense_retrieval", i)

if embedding_model_for_retrieval is not None:
    del embedding_model_for_retrieval
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

set_resume_pointer("dense_retrieval", len(SAMPLED_DATASET) - 1)
mark_section_complete("dense_retrieval")
print("✓ Checkpoint saved successfully (per-question files in retrieval/).")

# ------------------------------------------------------------
# Sanity checks + intermediate output
# ------------------------------------------------------------
assert len(RETRIEVAL_RESULTS) == len(SAMPLED_DATASET), "Missing retrieval results for some questions."
assert all(len(r["retrieved_chunks"]) > 0 for r in RETRIEVAL_RESULTS.values()), \
    "Some questions retrieved zero chunks — check FAISS scoping logic."

avg_retrieved = np.mean([len(r["retrieved_chunks"]) for r in RETRIEVAL_RESULTS.values()])
avg_top1_score = np.mean([r["retrieved_chunks"][0]["similarity_score"] for r in RETRIEVAL_RESULTS.values()])

print("\n" + "=" * 60)
print("SECTION 7 COMPLETE — DENSE RETRIEVAL SUMMARY")
print("=" * 60)
print(f"Questions processed         : {len(RETRIEVAL_RESULTS)}")
print(f"Configured top-k            : {CONFIG['top_k_retrieval']}")
print(f"Avg chunks retrieved/query  : {avg_retrieved:.2f}")
print(f"Avg top-1 similarity score  : {avg_top1_score:.4f}")
print("=" * 60 + "\n")

[2026-08-05 14:16:55] INFO: 50 / 50 questions need retrieval.
[2026-08-05 14:16:56] INFO: Loading SentenceTransformer model from sentence-transformers/all-mpnet-base-v2.
Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 31.70it/s]

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 62.27it/s]

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 56.58it/s]

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 47.02it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████

✓ Checkpoint saved successfully (per-question files in retrieval/).

SECTION 7 COMPLETE — DENSE RETRIEVAL SUMMARY
Questions processed         : 50
Configured top-k            : 8
Avg chunks retrieved/query  : 8.00
Avg top-1 similarity score  : 0.6568



In [10]:
# ============================================================
# SECTION 8: CROSS-ENCODER RERANKING
# ============================================================

def rerank_chunks_for_question(
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
    cross_encoder: CrossEncoder,
    rerank_top_k: int,
) -> List[Dict[str, Any]]:
    """
    Rerank a question's dense-retrieved chunks using a cross-encoder,
    keeping the top `rerank_top_k` by cross-encoder relevance score.

    Args:
        question: the question text
        retrieved_chunks: list of chunk dicts from Section 7's dense retrieval
        cross_encoder: loaded CrossEncoder model
        rerank_top_k: number of chunks to keep after reranking

    Returns:
        List of chunk dicts (with an added "rerank_score" field), sorted
        descending by rerank_score, truncated to rerank_top_k.

    Note:
        Shared identically by every baseline — the only place reranking
        happens in the notebook.
    """
    if not retrieved_chunks:
        return []
    pairs = [(question, chunk["text"]) for chunk in retrieved_chunks]
    rerank_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    for chunk, score in zip(retrieved_chunks, rerank_scores):
        chunk["rerank_score"] = float(score)
    reranked = sorted(retrieved_chunks, key=lambda c: c["rerank_score"], reverse=True)
    return reranked[:rerank_top_k]


def save_reranked_result(result: Dict[str, Any], path: Path) -> None:
    """Persist a single question's reranked result to a pickle file."""
    with open(path, "wb") as f:
        pickle.dump(result, f)


def load_reranked_result(path: Path) -> Dict[str, Any]:
    """Load a previously saved single-question reranked result."""
    with open(path, "rb") as f:
        return pickle.load(f)


# ------------------------------------------------------------
# Determine which questions still need reranking
# ------------------------------------------------------------
questions_needing_rerank = [
    r for r in SAMPLED_DATASET if not (retrieval_dir / f"{r['question_id']}_reranked.pkl").exists()
]

if not questions_needing_rerank and is_section_complete("cross_encoder_reranking"):
    print("✓ Existing checkpoint found for all questions.")
    cross_encoder_model = None
else:
    logger.info(f"{len(questions_needing_rerank)} / {len(SAMPLED_DATASET)} questions need reranking.")
    cross_encoder_model = CrossEncoder(CONFIG["cross_encoder_model"], device=CONFIG["device"])

# ------------------------------------------------------------
# Run reranking per-question, with per-question checkpointing
# ------------------------------------------------------------
RERANKED_RESULTS: Dict[str, Dict[str, Any]] = {}

for i, record in enumerate(tqdm(SAMPLED_DATASET, desc="Cross-encoder reranking per question")):
    qid = record["question_id"]
    result_path = retrieval_dir / f"{qid}_reranked.pkl"

    if result_path.exists():
        result = load_reranked_result(result_path)
    else:
        dense_result = RETRIEVAL_RESULTS[qid]
        reranked_chunks = rerank_chunks_for_question(
            question=dense_result["question"],
            retrieved_chunks=[dict(c) for c in dense_result["retrieved_chunks"]],
            cross_encoder=cross_encoder_model,
            rerank_top_k=CONFIG["rerank_top_k"],
        )
        result = {"question_id": qid, "question": dense_result["question"], "reranked_chunks": reranked_chunks}
        save_reranked_result(result, result_path)

    RERANKED_RESULTS[qid] = result

    if (i + 1) % checkpoint_every == 0:
        set_resume_pointer("cross_encoder_reranking", i)

if cross_encoder_model is not None:
    del cross_encoder_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

set_resume_pointer("cross_encoder_reranking", len(SAMPLED_DATASET) - 1)
mark_section_complete("cross_encoder_reranking")
print("✓ Checkpoint saved successfully (per-question files in retrieval/).")

# ------------------------------------------------------------
# Sanity checks + intermediate output
# ------------------------------------------------------------
assert len(RERANKED_RESULTS) == len(SAMPLED_DATASET), "Missing reranked results for some questions."
assert all(len(r["reranked_chunks"]) > 0 for r in RERANKED_RESULTS.values()), \
    "Some questions have zero reranked chunks."
assert all(len(r["reranked_chunks"]) <= CONFIG["rerank_top_k"] for r in RERANKED_RESULTS.values()), \
    "Reranked chunk count exceeds configured rerank_top_k."

avg_reranked = np.mean([len(r["reranked_chunks"]) for r in RERANKED_RESULTS.values()])
avg_top1_rerank_score = np.mean([r["reranked_chunks"][0]["rerank_score"] for r in RERANKED_RESULTS.values()])

print("\n" + "=" * 60)
print("SECTION 8 COMPLETE — CROSS-ENCODER RERANKING SUMMARY")
print("=" * 60)
print(f"Questions processed          : {len(RERANKED_RESULTS)}")
print(f"Configured rerank_top_k      : {CONFIG['rerank_top_k']}")
print(f"Avg chunks kept/query        : {avg_reranked:.2f}")
print(f"Avg top-1 rerank score       : {avg_top1_rerank_score:.4f}")
print("=" * 60 + "\n")

[2026-08-05 14:18:29] INFO: 50 / 50 questions need reranking.
[2026-08-05 14:18:29] INFO: No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
Cross-encoder reranking per question: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 24.31it/s]

✓ Checkpoint saved successfully (per-question files in retrieval/).

SECTION 8 COMPLETE — CROSS-ENCODER RERANKING SUMMARY
Questions processed          : 50
Configured rerank_top_k      : 4
Avg chunks kept/query        : 4.00
Avg top-1 rerank score       : 5.7685



In [11]:
# ============================================================
# SECTION 9: GENERATOR LLM + MODULAR BaseRAG INTERFACE
# ============================================================

from abc import ABC, abstractmethod


# ------------------------------------------------------------
# 9.1 — Generator model loading
# ------------------------------------------------------------
def load_generator_model(config: Dict[str, Any]) -> Tuple[AutoTokenizer, AutoModelForCausalLM]:
    """
    Load the shared generator LLM (tokenizer + model) used identically by
    every baseline in this notebook.

    Args:
        config: global CONFIG dict (uses generator_model, device,
                 generator_load_in_4bit)

    Returns:
        (tokenizer, model) tuple, model in eval mode on the configured device.
    """
    model_name = config["generator_model"]
    logger.info(f"Loading generator model: {model_name} ...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        # Many causal LMs (including Qwen2.5) ship without a pad token —
        # generation without one raises/silently misbehaves under newer
        # transformers versions.
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = None
    if config.get("generator_load_in_4bit") and config["device"] == "cuda":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16 if config["device"] == "cuda" else torch.float32,
        device_map=config["device"] if quant_config is None else "auto",
    )
    model.eval()
    logger.info(f"✓ Generator model loaded on {config['device']} "
                f"(4-bit={quant_config is not None})")
    return tokenizer, model


def generate_text(
    prompt: str,
    tokenizer: AutoTokenizer,
    model: AutoModelForCausalLM,
    config: Dict[str, Any],
    max_new_tokens: Optional[int] = None,
) -> str:
    """
    Generate a text completion for a single prompt using the shared
    generator LLM, applying the model's chat template.

    Args:
        prompt: user-turn prompt text (chat-templated internally)
        tokenizer: loaded tokenizer (with pad_token set)
        model: loaded causal LM
        config: global CONFIG dict (uses generator_temperature)
        max_new_tokens: override for CONFIG["generator_max_new_tokens"]

    Returns:
        Decoded generated text (new tokens only — the prompt is stripped
        out, avoiding the "echoes the prompt back" issue seen under some
        transformers versions when relying on skip_special_tokens alone).

    Note:
        This merges the transformers 5.x generation fix from your original
        notebook: explicit attention_mask, explicit pad_token_id, and
        slicing outputs to only the newly generated tokens rather than
        trusting decode() to drop the prompt.
    """
    messages = [{"role": "user", "content": prompt}]
    chat_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat_input, return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],       # required explicitly — was
                                                             # silently defaulting/breaking
                                                             # under transformers 5.x
            max_new_tokens=max_new_tokens or config["generator_max_new_tokens"],
            do_sample=config["generator_temperature"] > 0,
            temperature=max(config["generator_temperature"], 1e-5),
            pad_token_id=tokenizer.pad_token_id,             # required explicitly
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]  # strip prompt, decode only new tokens
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ------------------------------------------------------------
# Load once, share across every baseline for the rest of the notebook
# ------------------------------------------------------------
GENERATOR_TOKENIZER, GENERATOR_MODEL = load_generator_model(CONFIG)


# ------------------------------------------------------------
# 9.2 — Modular BaseRAG interface
# ------------------------------------------------------------
class BaseRAG(ABC):
    """
    Common interface every RAG baseline in this notebook must implement.

    `retrieve()` is intentionally NOT abstract — it is the fair-comparison
    enforcement mechanism. Every baseline retrieves via the exact same
    dense-retrieval + cross-encoder-reranking pipeline (Sections 7-8), so a
    baseline cannot quietly gain an advantage (or be handicapped) by
    retrieving differently. Only `verify()` and `answer()` — the genuine
    experimental variable — differ per baseline.

    Subclasses implement:
        verify(question, chunks) -> a verification/filtering decision
            specific to that baseline's strategy (may be a no-op for
            Vanilla RAG).
        answer(question, chunks) -> the final answer string, using the
            shared generator LLM via generate_text().

    All subclasses share:
        - the same reranked chunks for round 1 (from Section 8)
        - the same generator model/tokenizer instance (Section 9.1)
        - the same retrieval_budget_max_calls / max_iterations ceiling
          (CONFIG) if they perform additional retrieval rounds
    """

    def __init__(
        self,
        name: str,
        config: Dict[str, Any],
        tokenizer: AutoTokenizer,
        model: AutoModelForCausalLM,
        faiss_index: faiss.Index,
        embedding_model_getter,
        question_row_map: Dict[str, List[int]],
        chunk_metadata: List[Dict[str, Any]],
        chunk_texts: List[str],
        cross_encoder_getter,
    ) -> None:
        """
        Args:
            name: baseline identifier, used in results/checkpoints (e.g. "self_rag")
            config: global CONFIG dict
            tokenizer, model: shared generator LLM
            faiss_index, question_row_map, chunk_metadata, chunk_texts:
                shared retrieval artifacts from Sections 6-7
            embedding_model_getter: zero-arg callable returning a loaded
                SentenceTransformer (lazy — only invoked if a baseline
                actually needs an extra retrieval round beyond round 1)
            cross_encoder_getter: zero-arg callable returning a loaded
                CrossEncoder (lazy, same reasoning)
        """
        self.name = name
        self.config = config
        self.tokenizer = tokenizer
        self.model = model
        self.faiss_index = faiss_index
        self._embedding_model_getter = embedding_model_getter
        self._cross_encoder_getter = cross_encoder_getter
        self.question_row_map = question_row_map
        self.chunk_metadata = chunk_metadata
        self.chunk_texts = chunk_texts

    def retrieve(self, question_id: str, question: str, round_num: int = 1) -> List[Dict[str, Any]]:
        """
        Shared, non-overridable retrieval step. Round 1 reuses the
        precomputed Section 8 reranked chunks (no redundant compute).
        Rounds >1 (used by CRAG / Adaptive RAG / AdaKG-RAG multi-round /
        CalibRAG's verification-guided loop) perform a fresh
        dense-retrieve + rerank call, still through the exact same
        Section 7/8 functions.

        Args:
            question_id: HotpotQA question identifier
            question: question text (or a reformulated query string for
                round > 1)
            round_num: 1 for the initial retrieval, >1 for follow-up rounds

        Returns:
            List of reranked chunk dicts, identical in shape to Section 8's
            output.
        """
        if round_num == 1:
            return [dict(c) for c in RERANKED_RESULTS[question_id]["reranked_chunks"]]

        dense_result = retrieve_top_k_for_question(
            question=question, question_id=question_id,
            faiss_index=self.faiss_index, embedding_model=self._embedding_model_getter(),
            question_row_map=self.question_row_map, chunk_metadata=self.chunk_metadata,
            chunk_texts=self.chunk_texts, top_k=self.config["top_k_retrieval"],
        )
        return rerank_chunks_for_question(
            question=question, retrieved_chunks=dense_result["retrieved_chunks"],
            cross_encoder=self._cross_encoder_getter(), rerank_top_k=self.config["rerank_top_k"],
        )

    @abstractmethod
    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Baseline-specific verification/filtering step.

        Args:
            question: question text
            chunks: reranked chunks from retrieve()

        Returns:
            A dict describing the verification outcome. Shape is
            baseline-specific but MUST include at minimum:
                "verified_chunks": List[Dict] — chunks judged sufficient/relevant
                "needs_more_retrieval": bool — whether another round is warranted
                "trace": Any — baseline-specific diagnostic info (for logs)
        """
        raise NotImplementedError

    @abstractmethod
    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """
        Generate the final answer string given verified/selected chunks.

        Args:
            question: question text
            verified_chunks: chunks selected by verify()

        Returns:
            Final answer string.
        """
        raise NotImplementedError

    def run(self, record: Dict[str, Any]) -> Dict[str, Any]:
        """
        Orchestrate retrieve -> verify -> (optional extra rounds) -> answer
        for a single question, enforcing the shared retrieval-budget and
        max-iterations ceiling from CONFIG so no baseline can exceed the
        compute budget any other baseline is allowed.

        Args:
            record: a SAMPLED_DATASET question record

        Returns:
            Dict with question_id, predicted_answer, retrieval_rounds_used,
            latency_seconds, verification_trace, chunks_used.
        """
        question_id, question = record["question_id"], record["question"]
        start_time = time.time()

        round_num = 1
        chunks = self.retrieve(question_id, question, round_num=round_num)
        verification = self.verify(question, chunks)

        max_rounds = self.config["retrieval_budget_max_calls"]
        traces = [verification.get("trace")]

        while (
            verification.get("needs_more_retrieval", False)
            and round_num < max_rounds
        ):
            round_num += 1
            chunks = self.retrieve(question_id, question, round_num=round_num)
            verification = self.verify(question, chunks)
            traces.append(verification.get("trace"))

        predicted_answer = self.answer(question, verification["verified_chunks"])
        latency = time.time() - start_time

        return {
            "question_id": question_id,
            "baseline": self.name,
            "predicted_answer": predicted_answer,
            "retrieval_rounds_used": round_num,
            "latency_seconds": latency,
            "verification_trace": traces,
            "chunks_used": [c["chunk_id"] for c in verification["verified_chunks"]],
        }


logger.info("✓ Section 9 complete: generator loaded, BaseRAG interface defined.")
print(f"✓ Generator model loaded: {CONFIG['generator_model']} on {CONFIG['device']}")
print("✓ BaseRAG interface ready — Section 10 will implement each baseline against it.")

[2026-08-05 14:18:49] INFO: Loading generator model: Qwen/Qwen2.5-1.5B-Instruct ...
`torch_dtype` is deprecated! Use `dtype` instead!
[2026-08-05 14:19:55] INFO: ✓ Generator model loaded on cuda (4-bit=True)
[2026-08-05 14:19:55] INFO: ✓ Section 9 complete: generator loaded, BaseRAG interface defined.


✓ Generator model loaded: Qwen/Qwen2.5-1.5B-Instruct on cuda
✓ BaseRAG interface ready — Section 10 will implement each baseline against it.


In [12]:
# ============================================================
# SECTION 10a: BASELINES — VANILLA RAG (A) & AdaKG-RAG (E)
# ============================================================

BASELINE_REGISTRY: Dict[str, BaseRAG] = {}


def _lazy_embedding_model() -> SentenceTransformer:
    """Lazily load (once) the embedding model, for baselines needing extra retrieval rounds."""
    global _LAZY_EMBEDDING_MODEL_CACHE
    if "_LAZY_EMBEDDING_MODEL_CACHE" not in globals() or _LAZY_EMBEDDING_MODEL_CACHE is None:
        globals()["_LAZY_EMBEDDING_MODEL_CACHE"] = SentenceTransformer(
            CONFIG["embedding_model"], device=CONFIG["device"]
        )
    return _LAZY_EMBEDDING_MODEL_CACHE


def _lazy_cross_encoder() -> CrossEncoder:
    """Lazily load (once) the cross-encoder, for baselines needing extra retrieval rounds."""
    global _LAZY_CROSS_ENCODER_CACHE
    if "_LAZY_CROSS_ENCODER_CACHE" not in globals() or _LAZY_CROSS_ENCODER_CACHE is None:
        globals()["_LAZY_CROSS_ENCODER_CACHE"] = CrossEncoder(
            CONFIG["cross_encoder_model"], device=CONFIG["device"]
        )
    return _LAZY_CROSS_ENCODER_CACHE


def _build_context_block(chunks: List[Dict[str, Any]]) -> str:
    """Format a list of chunks into a numbered context block for prompting."""
    return "\n".join(f"[{i+1}] ({c['doc_title']}) {c['text']}" for i, c in enumerate(chunks))


def _parse_confidence_score(text: str) -> float:
    """
    Extract a self-reported confidence score in [0, 1] from free-form LLM
    output. Accepts "0.82", "82%", "82/100" style formats; falls back to
    0.5 (neutral) if nothing parseable is found, rather than raising, since
    a malformed self-report is itself meaningful baseline behavior, not an
    error condition.

    Args:
        text: raw generated text expected to contain a confidence figure.

    Returns:
        Float confidence in [0.0, 1.0].
    """
    import re
    percent_match = re.search(r"(\d{1,3}(?:\.\d+)?)\s*%", text)
    if percent_match:
        return max(0.0, min(1.0, float(percent_match.group(1)) / 100.0))

    decimal_match = re.search(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b", text)
    if decimal_match:
        return max(0.0, min(1.0, float(decimal_match.group(1))))

    return 0.5


# ------------------------------------------------------------
# Vanilla RAG (Category A)
# ------------------------------------------------------------
class VanillaRAG(BaseRAG):
    """
    Simplest possible baseline: retrieve once, generate once, no
    verification. Establishes the "no iterative verification" reference
    point required by the prompt's Category A.
    """

    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """No-op verification — Vanilla RAG accepts whatever was retrieved."""
        return {"verified_chunks": chunks, "needs_more_retrieval": False, "trace": None}

    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """Generate a direct answer conditioned on the retrieved context, no critique step."""
        context_block = _build_context_block(verified_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"Be concise — respond with just the answer, no explanation.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
        )
        return generate_text(prompt, self.tokenizer, self.model, self.config)


# ------------------------------------------------------------
# AdaKG-RAG (Category E — direct baseline paper)
# ------------------------------------------------------------
class AdaKGRAG(BaseRAG):
    """
    Reproduction of AdaKG-RAG's verification strategy: the LLM self-reports
    a confidence score for whether the retrieved context sufficiently
    supports an answer. If self-reported confidence falls below
    `adakg_confidence_threshold`, another retrieval round is triggered
    (up to the shared retrieval budget) using the low-confidence answer's
    key terms as a reformulated query.

    This is precisely the mechanism CalibRAG's claim-level NLI verification
    (Section 12+) is designed to replace: self-reported confidence is not
    mathematically grounded, whereas NLI entailment scores are.
    """

    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Ask the generator LLM to (a) draft a tentative answer and (b)
        self-report a confidence score for that answer given the context.

        Returns:
            trace contains the tentative answer and parsed confidence,
            for later inspection/logging.
        """
        context_block = _build_context_block(chunks)
        prompt = (
            f"Context:\n{context_block}\n\nQuestion: {question}\n\n"
            f"First, give a tentative answer. Then, on a new line, state your "
            f"confidence that this answer is fully supported by the context, "
            f"as a number between 0 and 1.\n"
            f"Format:\nAnswer: <answer>\nConfidence: <number>"
        )
        response = generate_text(prompt, self.tokenizer, self.model, self.config)
        confidence = _parse_confidence_score(response)

        needs_more = confidence < self.config["adakg_confidence_threshold"]
        return {
            "verified_chunks": chunks,
            "needs_more_retrieval": needs_more,
            "trace": {"tentative_response": response, "self_reported_confidence": confidence},
        }

    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """Final answer generation after the verification loop settles (confidence-accepted round)."""
        context_block = _build_context_block(verified_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"Be concise — respond with just the answer, no explanation.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
        )
        return generate_text(prompt, self.tokenizer, self.model, self.config)


def _instantiate_baseline(cls, name: str) -> BaseRAG:
    """Helper to construct a BaseRAG subclass with all shared dependencies wired in."""
    return cls(
        name=name, config=CONFIG, tokenizer=GENERATOR_TOKENIZER, model=GENERATOR_MODEL,
        faiss_index=FAISS_INDEX, embedding_model_getter=_lazy_embedding_model,
        question_row_map=QUESTION_ROW_MAP, chunk_metadata=CHUNK_METADATA_FROM_EMB,
        chunk_texts=CHUNK_TEXTS, cross_encoder_getter=_lazy_cross_encoder,
    )


BASELINE_REGISTRY["vanilla_rag"] = _instantiate_baseline(VanillaRAG, "vanilla_rag")
BASELINE_REGISTRY["adakg_rag"] = _instantiate_baseline(AdaKGRAG, "adakg_rag")

print("✓ Registered baselines:", list(BASELINE_REGISTRY.keys()))

✓ Registered baselines: ['vanilla_rag', 'adakg_rag']


In [13]:
# ============================================================
# SECTION 10b: BASELINES — Self-RAG (B) & CRAG (C)
# ============================================================
#
# ADAPTATION NOTE (document this in the paper's baseline/limitations section):
# Self-RAG's original paper trains a dedicated critic model to emit reflection
# tokens (ISREL/ISSUP/ISUSE); CRAG's original paper trains a lightweight
# retrieval evaluator and may fall back to live web search. Both are
# reproduced here using the SAME shared generator LLM as critic/evaluator
# (no separate fine-tuning), and CRAG's corrective action re-queries the
# same fixed per-question context rather than the open web — required to
# keep "identical retriever, identical retrieval budget" across baselines.

def _parse_yes_no(text: str, default: bool = False) -> bool:
    """Parse a yes/no-style critique response into a boolean, defaulting on ambiguity."""
    normalized = text.strip().lower()
    if normalized.startswith("yes"):
        return True
    if normalized.startswith("no"):
        return False
    return default


# ------------------------------------------------------------
# Self-RAG (Category B)
# ------------------------------------------------------------
class SelfRAG(BaseRAG):
    """
    Self-reflective retrieval baseline. Uses the shared generator LLM as
    its own critic for two reflection steps:
      1. ISREL-equivalent: is each retrieved chunk relevant to the question?
      2. ISSUP-equivalent: is the drafted answer supported by the filtered,
         relevant chunks?
    If support fails, retrieves again (up to the shared budget).
    """

    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Critique each chunk for relevance (ISREL-equivalent), then critique
        a draft answer for support (ISSUP-equivalent) using only the
        relevant chunks.
        """
        relevant_chunks = []
        relevance_votes = []
        for chunk in chunks:
            prompt = (
                f"Question: {question}\nPassage: {chunk['text']}\n\n"
                f"Is this passage relevant to answering the question? Answer only Yes or No."
            )
            response = generate_text(prompt, self.tokenizer, self.model, self.config, max_new_tokens=8)
            is_relevant = _parse_yes_no(response, default=True)  # default-keep on ambiguous critique
            relevance_votes.append(is_relevant)
            if is_relevant:
                relevant_chunks.append(chunk)

        if not relevant_chunks:
            relevant_chunks = chunks  # avoid an empty context if the critic rejects everything

        context_block = _build_context_block(relevant_chunks)
        draft_prompt = (
            f"Context:\n{context_block}\n\nQuestion: {question}\n"
            f"Give a concise draft answer using only this context.\nAnswer:"
        )
        draft_answer = generate_text(draft_prompt, self.tokenizer, self.model, self.config)

        support_prompt = (
            f"Context:\n{context_block}\n\nDraft answer: {draft_answer}\n\n"
            f"Is the draft answer fully supported by the context above? Answer only Yes or No."
        )
        support_response = generate_text(support_prompt, self.tokenizer, self.model, self.config, max_new_tokens=8)
        is_supported = _parse_yes_no(support_response, default=False)

        return {
            "verified_chunks": relevant_chunks,
            "needs_more_retrieval": not is_supported,
            "trace": {
                "relevance_votes": relevance_votes,
                "draft_answer": draft_answer,
                "is_supported": is_supported,
            },
        }

    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """Final answer generation using only ISREL-filtered, ISSUP-accepted chunks."""
        context_block = _build_context_block(verified_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"Be concise — respond with just the answer, no explanation.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
        )
        return generate_text(prompt, self.tokenizer, self.model, self.config)


# ------------------------------------------------------------
# CRAG (Category C)
# ------------------------------------------------------------
class CRAG(BaseRAG):
    """
    Corrective RAG baseline. A retrieval evaluator classifies the retrieved
    set as Correct / Incorrect / Ambiguous using the mean cross-encoder
    rerank score (already computed at retrieval time — consistent with
    CRAG's own strip-level scoring design), then takes the corresponding
    corrective action.
    """

    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Classify retrieval quality via mean rerank score, then apply the
        matching CRAG corrective action:
          - Correct    -> knowledge refinement (keep only above-median strips)
          - Incorrect  -> discard, flag for re-retrieval with a reformulated query
          - Ambiguous  -> refine existing AND flag for re-retrieval (CRAG combines both)
        """
        scores = [c["rerank_score"] for c in chunks]
        mean_score = float(np.mean(scores)) if scores else 0.0

        if mean_score >= self.config["crag_correct_threshold"]:
            verdict = "Correct"
        elif mean_score < self.config["crag_incorrect_threshold"]:
            verdict = "Incorrect"
        else:
            verdict = "Ambiguous"

        median_score = float(np.median(scores)) if scores else 0.0
        refined_chunks = [c for c in chunks if c["rerank_score"] >= median_score] or chunks

        if verdict == "Correct":
            verified_chunks, needs_more = refined_chunks, False
        elif verdict == "Incorrect":
            verified_chunks, needs_more = chunks, True   # discard refinement, force re-retrieval
        else:  # Ambiguous
            verified_chunks, needs_more = refined_chunks, True

        return {
            "verified_chunks": verified_chunks,
            "needs_more_retrieval": needs_more,
            "trace": {"verdict": verdict, "mean_rerank_score": mean_score},
        }

    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """Final answer generation using CRAG's refined/corrected chunk set."""
        context_block = _build_context_block(verified_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"Be concise — respond with just the answer, no explanation.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
        )
        return generate_text(prompt, self.tokenizer, self.model, self.config)


BASELINE_REGISTRY["self_rag"] = _instantiate_baseline(SelfRAG, "self_rag")
BASELINE_REGISTRY["crag"] = _instantiate_baseline(CRAG, "crag")

print("✓ Registered baselines:", list(BASELINE_REGISTRY.keys()))

✓ Registered baselines: ['vanilla_rag', 'adakg_rag', 'self_rag', 'crag']


In [14]:
# ============================================================
# SECTION 10c: BASELINE — Adaptive RAG (Category D)
# ============================================================
#
# ADAPTATION NOTE (same as Self-RAG/CRAG): the original paper's complexity
# classifier is a small trained model; here the shared generator LLM performs
# this classification via prompting instead. Documented in CONFIG as
# adaptive_rag_complexity_model="generator".

def _classify_query_complexity(question: str, tokenizer, model, config: Dict[str, Any]) -> str:
    """
    Classify a question's retrieval complexity using the shared generator LLM.

    Args:
        question: question text
        tokenizer, model: shared generator LLM
        config: global CONFIG dict

    Returns:
        One of "no_retrieval", "single_step", "multi_step". Defaults to
        "single_step" (the safe middle ground) if the LLM's output doesn't
        cleanly match a known label.
    """
    prompt = (
        f"Question: {question}\n\n"
        f"Classify how much external evidence retrieval this question needs:\n"
        f"- NO_RETRIEVAL: answerable from general knowledge alone, no lookup needed\n"
        f"- SINGLE_STEP: needs exactly one lookup of a single fact\n"
        f"- MULTI_STEP: needs multiple, chained lookups across different facts "
        f"(e.g. multi-hop reasoning)\n\n"
        f"Respond with only one label: NO_RETRIEVAL, SINGLE_STEP, or MULTI_STEP."
    )
    response = generate_text(prompt, tokenizer, model, config, max_new_tokens=10).strip().upper()

    if "NO_RETRIEVAL" in response:
        return "no_retrieval"
    if "MULTI_STEP" in response:
        return "multi_step"
    if "SINGLE_STEP" in response:
        return "single_step"
    return "single_step"  # safe default on ambiguous classification


class AdaptiveRAG(BaseRAG):
    """
    Complexity-adaptive baseline. Routes each question through no-retrieval,
    single-step, or multi-step retrieval based on an LLM-driven complexity
    classification (see module note above for the adaptation from a trained
    classifier to a prompted one).
    """

    def verify(self, question: str, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Note: for AdaptiveRAG, complexity classification (not chunk
        verification) is the real decision point, and it happens BEFORE
        BaseRAG.run() calls retrieve() at all — see run() override below.
        This verify() only governs the multi_step loop's stopping condition
        once retrieval has actually happened.
        """
        context_block = _build_context_block(chunks)
        draft_prompt = (
            f"Context:\n{context_block}\n\nQuestion: {question}\n"
            f"Give a concise draft answer using only this context.\nAnswer:"
        )
        draft_answer = generate_text(draft_prompt, self.tokenizer, self.model, self.config)

        support_prompt = (
            f"Context:\n{context_block}\n\nDraft answer: {draft_answer}\n\n"
            f"Is the draft answer fully supported by the context above? Answer only Yes or No."
        )
        support_response = generate_text(support_prompt, self.tokenizer, self.model, self.config, max_new_tokens=8)
        is_supported = _parse_yes_no(support_response, default=True)

        # Only the multi_step route is allowed to request another round —
        # enforced in run() below via self._current_complexity.
        needs_more = (not is_supported) and (self._current_complexity == "multi_step")
        return {
            "verified_chunks": chunks,
            "needs_more_retrieval": needs_more,
            "trace": {"draft_answer": draft_answer, "is_supported": is_supported},
        }

    def answer(self, question: str, verified_chunks: List[Dict[str, Any]]) -> str:
        """Final answer — falls back to parametric-only generation if no_retrieval was chosen."""
        if self._current_complexity == "no_retrieval":
            prompt = f"Answer this question concisely using your own knowledge.\nQuestion: {question}\nAnswer:"
            return generate_text(prompt, self.tokenizer, self.model, self.config)

        context_block = _build_context_block(verified_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"Be concise — respond with just the answer, no explanation.\n\n"
            f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
        )
        return generate_text(prompt, self.tokenizer, self.model, self.config)

    def run(self, record: Dict[str, Any]) -> Dict[str, Any]:
        """
        Override BaseRAG.run() to classify complexity FIRST, then either
        skip retrieval entirely (no_retrieval) or defer to the standard
        retrieve->verify->answer loop (single_step / multi_step), still
        respecting the shared retrieval budget for multi_step.
        """
        question_id, question = record["question_id"], record["question"]
        start_time = time.time()

        self._current_complexity = _classify_query_complexity(
            question, self.tokenizer, self.model, self.config
        )

        if self._current_complexity == "no_retrieval":
            predicted_answer = self.answer(question, verified_chunks=[])
            return {
                "question_id": question_id,
                "baseline": self.name,
                "predicted_answer": predicted_answer,
                "retrieval_rounds_used": 0,
                "latency_seconds": time.time() - start_time,
                "verification_trace": [{"complexity": "no_retrieval"}],
                "chunks_used": [],
            }

        # single_step / multi_step: defer to the standard BaseRAG loop,
        # which already respects retrieval_budget_max_calls.
        result = super().run(record)
        result["verification_trace"].insert(0, {"complexity": self._current_complexity})
        return result


BASELINE_REGISTRY["adaptive_rag"] = _instantiate_baseline(AdaptiveRAG, "adaptive_rag")

print("✓ Registered baselines:", list(BASELINE_REGISTRY.keys()))

✓ Registered baselines: ['vanilla_rag', 'adakg_rag', 'self_rag', 'crag', 'adaptive_rag']
